# Бинарная классификация "таблица - не таблица"

## Подготовка размеченных данных

Данные были размечены инструментом CVAT

In [ ]:
import pandas as pd
df = pd.read_xml('annotations.xml')
df

Загрузить напрямую из XML не получилось. Попросил DeepSeek написать функцию для парсинга XML файла в словарь

In [ ]:
import xml.etree.ElementTree as ET

tree = ET.parse('annotations.xml')
root = tree.getroot()

In [ ]:
def etree_to_dict(t):
    d = {t.tag: {} if t.attrib else None}
    children = list(t)
    if children:
        dd = {}
        for dc in map(etree_to_dict, children):
            for k, v in dc.items():
                if k in dd:
                    if not isinstance(dd[k], list):
                        dd[k] = [dd[k]]
                    dd[k].append(v)
                else:
                    dd[k] = v
        d = {t.tag: dd}
    if t.attrib:
        d[t.tag].update(('@' + k, v) for k, v in t.attrib.items())
    if t.text and t.text.strip():
        if children or t.attrib:
            d[t.tag]['#text'] = t.text.strip()
        else:
            d[t.tag] = t.text.strip()
    return d

root = ET.parse('annotations.xml').getroot()
full_dict = etree_to_dict(root)

Выгрузка в JSON чтобы просмотреть получившийся словарь

In [ ]:
import json
with open('labels.json', 'w', encoding='utf8') as jsonfile:
    json.dump(full_dict, jsonfile, indent=4, ensure_ascii=False)

Так как тэг хранится во вложенном словаре, надо его оттуда вытащить. Так же некоторые фотографии остались неразмеченными, это тоже надо учесть

In [ ]:
df = pd.DataFrame(full_dict['annotations']['image'])
df['tag'] = df['tag'].apply(lambda x: x.get('@label') if isinstance(x, dict) else x)
df

Выгрузим полученный фрейм в Excel и заполним пропуски вручную

In [ ]:
df.to_excel('labels.xlsx')

## Feature Exctraction

Попробуем метод Hu Moments для задачи FE.

In [ ]:
import cv2
import numpy as np
import pandas as pd

Первый взгляд на работу с библиотекой CV2

In [ ]:
im = cv2.imread('data/photo_527@12-06-2023_13-31-42.jpg', cv2.IMREAD_GRAYSCALE)
im.shape

In [ ]:
_, im = cv2.threshold(im, 128, 255, cv2.THRESH_BINARY)

In [ ]:
m = cv2.moments(im)
hm = cv2.HuMoments(m)
hm

In [ ]:
for i in range(0,7):
   hm[i] = -1* np.copysign(1.0, hm[i]) * np.log10(abs(hm[i]))

Снова считаем датафрейм после заполнения пустых ячеек

In [ ]:
df = pd.read_excel('labels.xlsx')
df.columns

Функция для df.apply которая рассчитает Hu Moments

In [ ]:
def get_hu_moments(img_path):
    im = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    _, im = cv2.threshold(im, 128, 255, cv2.THRESH_BINARY)
    m = cv2.moments(im)
    hm = cv2.HuMoments(m)
    for i in range(0,7):
        hm[i] = -1* np.copysign(1.0, hm[i]) * np.log10(np.abs(hm[i]) + 1e-12)
    
    return [moment[0] for moment in hm]


In [ ]:
df['hu_moments'] = df['name'].apply(lambda x: get_hu_moments('data/' + x))

Функция возвращает список, поэтому разделим списки на колоник в датафрейме

In [ ]:
cols = [f'hm_{i}' for i in range(0, 7)]

In [ ]:
df[cols] = pd.DataFrame(
    df['hu_moments'].tolist(),
    index=df.index
)

In [ ]:
df

# Бинарная классификация

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

In [ ]:
X = df[cols]
y = df['tag']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
neigh = KNeighborsClassifier(n_neighbors=3)
neigh.fit(X_train, y_train)

In [ ]:
y_pred = neigh.predict(X_test)

In [107]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred, pos_label='table'))
print(classification_report(y_test, y_pred))

Accuracy: 0.9826338639652678
F1: 0.9795221843003413
              precision    recall  f1-score   support

   not_table       1.00      0.97      0.98       403
       table       0.96      1.00      0.98       288

    accuracy                           0.98       691
   macro avg       0.98      0.98      0.98       691
weighted avg       0.98      0.98      0.98       691

